# Numerical Sets — Set Transformer Experiments

This notebook compares five architectures on a numerical set regression task: predicting the maximum value from a set of 100 numbers.

Models evaluated:
- Standard Transformer
- Full Set Transformer
- ISAB + PMA
- SAB + PMA
- MLP Baseline

Additional experiments examine permutation invariance and data efficiency.

## 1. HPC Environment Setup

In [2]:
import os
from pathlib import Path
os.chdir("/work/classes/ai4200-001-2026f/jfinnegan42/Set-Transformer")
import sys
import subprocess
from pathlib import Path

result = subprocess.run(
    ["spack", "location", "-i", "py-torchvision@0.24.1"],
    capture_output=True,
    text=True
)

root = Path(result.stdout.strip())
site_packages = root / "lib" / "python3.13" / "site-packages"

sys.path.insert(0, str(site_packages))

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.9.1
Torchvision: 0.24.1
CUDA available: True
GPU: NVIDIA A100-PCIE-40GB


## 2. Model Training

In [3]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(sys.path)

command = [
    sys.executable,
    "train.py",
    "--dataset", "numerical",
    "--model", "all",
    "--epochs", "20"
]

result = subprocess.run(command, env=env)
print("Exit code:", result.returncode)

Device: cuda

STANDARD / NUMERICAL / 396,929 parameters
Epoch 01/20 | train loss 0.0644 | val loss 0.0022 | val MAE 0.0421
Epoch 02/20 | train loss 0.0011 | val loss 0.0058 | val MAE 0.0686
Epoch 03/20 | train loss 0.0006 | val loss 0.0152 | val MAE 0.1160
Epoch 04/20 | train loss 0.0005 | val loss 0.0091 | val MAE 0.0884
Epoch 05/20 | train loss 0.0004 | val loss 0.0126 | val MAE 0.1040
Epoch 06/20 | train loss 0.0004 | val loss 0.0125 | val MAE 0.1028
Epoch 07/20 | train loss 0.0003 | val loss 0.0123 | val MAE 0.1024
Epoch 08/20 | train loss 0.0003 | val loss 0.0116 | val MAE 0.1025
Epoch 09/20 | train loss 0.0002 | val loss 0.0138 | val MAE 0.1093
Epoch 10/20 | train loss 0.0002 | val loss 0.0102 | val MAE 0.0942
Epoch 11/20 | train loss 0.0002 | val loss 0.0127 | val MAE 0.1045
Epoch 12/20 | train loss 0.0002 | val loss 0.0150 | val MAE 0.1161
Epoch 13/20 | train loss 0.0001 | val loss 0.0132 | val MAE 0.1070
Epoch 14/20 | train loss 0.0001 | val loss 0.0143 | val MAE 0.1121
Epoch 

## 3. Training and Validation Loss

The following graphs compare all five architectures over 20 training epochs.

### Training Loss

![Training Loss](../results/numerical_training_loss.png)

### Validation Loss

![Validation Loss](../results/numerical_validation_loss.png)

### Final Test Performance

![Test MAE](../results/numerical_test_mae.png)

## 4. Loading Trained Models

In [8]:
import torch
from baseline import MLPBaseline
from transformer import StandardTransformer
from set_transformer import SetTransformer, ISAB_PMA, SAB_PMA
from load_data import load_NumericalSets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {
    "standard": StandardTransformer(),
    "set": SetTransformer(),
    "isab": ISAB_PMA(),
    "sab": SAB_PMA(),
    "baseline": MLPBaseline()
}

for name, model in models.items():
    checkpoint = torch.load(
        f"checkpoints/numerical_{name}.pt",
        map_location=device,
        weights_only=True
    )

    # Supports checkpoints containing either a state dictionary
    # or a dictionary with a model_state_dict entry.
    if "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]

    model.load_state_dict(checkpoint)
    model.to(device)
    model.eval()

    print(f"{name}: loaded successfully")

print("Device:", device)

standard: loaded successfully
set: loaded successfully
isab: loaded successfully
sab: loaded successfully
baseline: loaded successfully
Device: cuda


## 5. Permutation Invariance Test

In [9]:
import torch
from load_data import load_NumericalSets

_, test_set = load_NumericalSets()

# Use 100 test examples
X = torch.stack([test_set[i][0] for i in range(100)]).to(device)

# Randomly shuffle the 100 elements within each set
torch.manual_seed(42)
permutations = torch.rand(100, 100, device=device).argsort(dim=1)

X_shuffled = torch.gather(
    X,
    dim=1,
    index=permutations.unsqueeze(-1)
)

print("PERMUTATION INVARIANCE TEST")
print("-" * 60)

with torch.no_grad():
    for name, model in models.items():
        original = model(X).reshape(100, -1)
        shuffled = model(X_shuffled).reshape(100, -1)

        differences = (original - shuffled).abs()

        print(
            f"{name:10} | "
            f"Mean difference: {differences.mean().item():.8f} | "
            f"Max difference: {differences.max().item():.8f}"
        )

PERMUTATION INVARIANCE TEST
------------------------------------------------------------
standard   | Mean difference: 0.00000020 | Max difference: 0.00000072
set        | Mean difference: 0.00000018 | Max difference: 0.00000048
isab       | Mean difference: 0.00000018 | Max difference: 0.00000072
sab        | Mean difference: 0.00000012 | Max difference: 0.00000048
baseline   | Mean difference: 0.14949723 | Max difference: 0.62835133


## 6. Saving Permutation Test Results

In [10]:
import pandas as pd
from pathlib import Path

results = []

with torch.no_grad():
    for name, model in models.items():
        original = model(X).reshape(100, -1)
        shuffled = model(X_shuffled).reshape(100, -1)
        differences = (original - shuffled).abs()

        results.append({
            "model": name,
            "mean_difference": differences.mean().item(),
            "max_difference": differences.max().item()
        })

Path("results").mkdir(exist_ok=True)

df = pd.DataFrame(results)
df.to_csv("results/numerical_permutation_test.csv", index=False)

print(df)
print("Saved: results/numerical_permutation_test.csv")

      model  mean_difference  max_difference
0  standard     1.955032e-07    7.152557e-07
1       set     1.788139e-07    4.768372e-07
2      isab     1.847744e-07    7.152557e-07
3       sab     1.156330e-07    4.768372e-07
4  baseline     1.494972e-01    6.283513e-01
Saved: results/numerical_permutation_test.csv


### Permutation Invariance Results

![Permutation Invariance](../results/numerical_permutation_invariance.png)

All four Transformer architectures produced nearly identical predictions when the elements of each input set were shuffled.

The MLP baseline showed a much larger mean prediction difference (0.1495), demonstrating its sensitivity to input ordering.

## 7. Data Efficiency Experiments

In [12]:
import subprocess
import sys
import os

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(sys.path)

fractions = [0.10, 0.25, 0.50, 1.00]
models_to_test = ["set", "baseline"]

for model in models_to_test:
    for fraction in fractions:
        print(f"\n{'=' * 60}", flush=True)
        print(f"Training {model.upper()} with {fraction:.0%} of data", flush=True)
        print(f"{'=' * 60}", flush=True)

        command = [
            sys.executable,
            "-u",
            "train.py",
            "--dataset", "numerical",
            "--model", model,
            "--epochs", "20",
            "--train-fraction", str(fraction)
        ]

        result = subprocess.run(command, env=env)

        if result.returncode != 0:
            print(f"FAILED: {model}, {fraction:.0%}", flush=True)
            break

print("\nData-efficiency training finished.", flush=True)


Training SET with 10% of data
Device: cuda
Training fraction: 10%
Training examples: 900
Validation examples: 1000
Test examples: 2000

SET / NUMERICAL / 422,017 parameters
Epoch 01/20 | train loss 0.5547 | val loss 0.4919 | val MAE 0.6169
Epoch 02/20 | train loss 0.2757 | val loss 0.2721 | val MAE 0.3939
Epoch 03/20 | train loss 0.1989 | val loss 0.1714 | val MAE 0.3243
Epoch 04/20 | train loss 0.2024 | val loss 0.1562 | val MAE 0.3082
Epoch 05/20 | train loss 0.1290 | val loss 0.0517 | val MAE 0.1856
Epoch 06/20 | train loss 0.0382 | val loss 0.0154 | val MAE 0.0989
Epoch 07/20 | train loss 0.0278 | val loss 0.0169 | val MAE 0.1099
Epoch 08/20 | train loss 0.0128 | val loss 0.0049 | val MAE 0.0573
Epoch 09/20 | train loss 0.0039 | val loss 0.0059 | val MAE 0.0659
Epoch 10/20 | train loss 0.0040 | val loss 0.0025 | val MAE 0.0381
Epoch 11/20 | train loss 0.0019 | val loss 0.0016 | val MAE 0.0311
Epoch 12/20 | train loss 0.0017 | val loss 0.0018 | val MAE 0.0352
Epoch 13/20 | train lo

## 8. Data Efficiency Results

The Full Set Transformer and MLP baseline were trained using 10%, 25%, 50%, and 100% of the available training data.

### Data Efficiency Comparison

![Data Efficiency](../results/numerical_data_efficiency.png)

The Full Set Transformer achieved substantially lower test mean absolute error (MAE) than the MLP baseline at every training-data fraction.

As more training data became available, the Set Transformer's error decreased, while the MLP baseline showed little improvement.